<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Prepare the Data for Building a Conversational Model

In this first lab, we will load, clean and format the data for training a conversational model.
After completing the following tasks:

1. Defining our conversational task,
2. Downloading or constructing our dataset with prompts and responses,
3. Documenting our dataset with a Data Card, and
4. Assessing the impact of your conversational model.

## Overview

In this lab, we will:

* Load the dataset.

* Clean individual datapoints, if necessary.

* Split the dataset into training and test sets.

* Format the data and prepare it for LLM fine-tuning.

* Save the dataset in a format that is compatible with the machine learning pipeline.

The end result of this lab will be a processed dataset that can be used in training and evaluating our conversational model.


## Imports

Import a range of libraries that will be used for preparing our dataset.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill # Text wrapping utilities.
from typing import Any, Dict, Tuple, List

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

from sklearn.model_selection import train_test_split # Data splitting utils.
import matplotlib.pyplot as plt # Plotting and visualisation.

# Deep learning and AI libraries.
import jax.numpy as jnp # JAX numerical computing.
import keras # High-level neural networks API.
import keras_nlp # Keras NLP extensions.

# Google Colab specific.
from google.colab import drive # Access to Google Drive.
from google.colab import userdata # Access to Colab secrets.

## Load the data

One method to do this is to upload the dataset to Google Drive and then connect your Google Drive to Colab.

### Connect Colab with Google Drive

The following cell mounts your Google Drive so that you can access files by adding `/content/drive/MyDrive/` to the beginning of your file path.


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


### 1: Load and inspect the data


## Merge datasets and prepare train/test splits

This cell combines the two generated datasets, then splits the data
correctly before building the final training pairs.

**What this cell does:**
* Merges the 100 hand-crafted (Opus 4.8) and 900 template-augmented
  (labeled Sonnet 5) conversation datasets into a single JSONL file.
* Loads the merged data into a DataFrame, where each row is one full
  conversation (with a `turns` list of student/tutor exchanges).
* **Splits into train/test at the conversation level, before creating
  any Q&A pairs**: splitting after flattening into
  pairs would let pieces of the same multi-turn conversation leak across
  both the train and test sets, making the test set no longer a fair,
  unseen evaluation. The split is stratified by `concept`, so all 11
  linear algebra topics remain proportionally represented in both sets.
* Builds Q&A pairs from each split *separately* via `build_pairs()`,
  which walks each conversation's turns two at a time. For multi-turn
  conversations, prior exchanges are carried forward as `history` so
  each pair still has the context it needs to make sense on its own,
  rather than being reduced to a lone, context-free student message.
* Displays the resulting shapes, columns, and a sample of pairs that
  include multi-turn history, to confirm everything built correctly
  before moving on to formatting and tokenization.

In [ ]:
file_path1 = "/content/drive/MyDrive/Colab Notebooks/tutor_dataset_100.jsonl"
file_path2 = "/content/drive/MyDrive/Colab Notebooks/tutor_dataset_900.jsonl"

file_list = [file_path1, file_path2]
new_file = "tutor_dataset.jsonl"

SOT = "<start_of_turn>"
EOT = "<end_of_turn>"

# Merge the 2 datasets generated by Opus 4.8 & Sonnet 5
with open(new_file, 'w', encoding='utf-8') as outfile:
    for file_name in file_list:
        # Open and read each source file
        with open(file_name, 'r', encoding='utf-8') as infile:
            outfile.write(infile.read())
            # Optional: Add a newline if files don't end with one
            outfile.write('\n')

df_before = pd.read_json(new_file, lines=True)

display(df_before.head())
print("\n")
display(df_before.tail())
print("\n")
display(df_before.info())
print("\n")
print("Columns:", df_before.columns.tolist())
print("Number of rows:", len(df_before))

# --------------------------------------------------
# Split into train/test BEFORE creating Q&A pairs, so no conversation's
# turns leak across both sides. Split on whole conversations (rows), and
# stratify by concept so all 11 topics stay balanced in both splits.
df_train, df_test = train_test_split(
    df_before,
    test_size=0.2,
    random_state=42,
    stratify=df_before["concept"],
)
print("\ntrain conversations:", len(df_train))
print("test conversations: ", len(df_test))
# --------------------------------------------------

# Extract Q&A pairs from a dataframe of conversations.
# Iterate each row's "turns" two at a time and build one pair per exchange.
# For multi-turn rows, fold prior exchanges into the student side so each
# pair still makes sense on its own.
def build_pairs(conv_df):
    pairs = []
    for _, row in conv_df.iterrows():
        turns = row["turns"]
        prior = []
        for i in range(0, len(turns) - 1, 2):
            student_msg = turns[i]["content"]
            tutor_msg = turns[i + 1]["content"]
            pairs.append({
                "history": list(prior),
                "input":   student_msg,
                "output":  tutor_msg,
            })
            # Add previous conversations to "prior"
            prior.append({"role": "student", "content": student_msg})
            prior.append({"role": "tutor",   "content": tutor_msg})
    return pd.DataFrame(pairs)

# Build pairs for each split SEPARATELY
print("\nQ&A pairs")
train_pairs = build_pairs(df_train)
test_pairs  = build_pairs(df_test)

display(train_pairs.shape)
display(train_pairs.head())
print("\n")
display(train_pairs.info())
print("\n")
display(train_pairs[train_pairs["history"] != ""].head())
print("\n")
print("Columns:", train_pairs.columns.tolist())
print("train pairs:", len(train_pairs), "| test pairs:", len(test_pairs))

,persona,concept,example_type,scaffolding_stage,visualization_technique,student_error,turns,source
0,patient_socratic_tutor,dot product,multi_turn,light,pattern_testing,False,"[{'role': 'student', 'content': 'I can compute...",opus
1,patient_socratic_tutor,determinants,single_turn,direct_answer,metaphor,False,"[{'role': 'student', 'content': 'I've been try...",opus
2,patient_socratic_tutor,matrix multiplication,single_turn,moderate,pattern_testing,True,"[{'role': 'student', 'content': 'I multiplied ...",opus
3,patient_socratic_tutor,eigenvalues/eigenvectors,multi_turn,moderate,metaphor,False,"[{'role': 'student', 'content': 'I don't get w...",opus
4,patient_socratic_tutor,vectors,single_turn,heavy,tangible_anchor,False,"[{'role': 'student', 'content': 'What's the di...",opus


,persona,concept,example_type,scaffolding_stage,visualization_technique,student_error,turns,source
995,patient_socratic_tutor,vectors,single_turn,direct_answer,tangible_anchor,False,"[{'role': 'student', 'content': 'Running out o...",sonnet
996,patient_socratic_tutor,eigenvalues/eigenvectors,single_turn,light,metaphor,True,"[{'role': 'student', 'content': 'For [[-3,0],[...",sonnet
997,patient_socratic_tutor,matrix multiplication,single_turn,heavy,metaphor,True,"[{'role': 'student', 'content': 'I worked out ...",sonnet
998,patient_socratic_tutor,rank,single_turn,moderate,metaphor,True,"[{'role': 'student', 'content': 'For [[-1,-6],...",sonnet
999,patient_socratic_tutor,span,single_turn,moderate,tangible_anchor,False,"[{'role': 'student', 'content': 'For [-7,-1] a...",sonnet




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   persona                  1000 non-null   object
 1   concept                  1000 non-null   object
 2   example_type             1000 non-null   object
 3   scaffolding_stage        1000 non-null   object
 4   visualization_technique  1000 non-null   object
 5   student_error            1000 non-null   bool  
 6   turns                    1000 non-null   object
 7   source                   1000 non-null   object
dtypes: bool(1), object(7)
memory usage: 55.8+ KB


None



Columns: ['persona', 'concept', 'example_type', 'scaffolding_stage', 'visualization_technique', 'student_error', 'turns', 'source']
Number of rows: 1000

train conversations: 800
test conversations:  200

Q&A pairs


...

,history,input,output
0,[],"Is [-43,29] right for [[3,-4],[-3,2]] times [-...","Right — [-43,29] is correct. Nice — can you sa..."
1,[],"Quick check on [5,4] and [-5,2]: am I supposed...",Picture it like every shade obtainable by mixi...
2,[],I'm exhausted — please just give me the answer...,It's 1: 5·5 − 8·3. Think of it as an area-grow...
3,[],"I really just need the result for [[3,4],[-2,3...","The answer is [-23,-30], from row one: 3·3 + 4..."
4,[],"So for [[4,-5],[-2,1]], whether one row is a s...",Right — 2 is correct. Right instinct — for [[4...




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   history  958 non-null    object
 1   input    958 non-null    object
 2   output   958 non-null    object
dtypes: object(3)
memory usage: 22.6+ KB


None

,history,input,output
0,[],"Is [-43,29] right for [[3,-4],[-3,2]] times [-...","Right — [-43,29] is correct. Nice — can you sa..."
1,[],"Quick check on [5,4] and [-5,2]: am I supposed...",Picture it like every shade obtainable by mixi...
2,[],I'm exhausted — please just give me the answer...,It's 1: 5·5 − 8·3. Think of it as an area-grow...
3,[],"I really just need the result for [[3,4],[-2,3...","The answer is [-23,-30], from row one: 3·3 + 4..."
4,[],"So for [[4,-5],[-2,1]], whether one row is a s...",Right — 2 is correct. Right instinct — for [[4...




Columns: ['history', 'input', 'output']
train pairs: 958 | test pairs: 238


### 2: Format the data


## Convert Q&A pairs into instruction records and save

This cell reshapes each Q&A pair into the format expected for training,
then writes both splits to disk for later use.

**What this cell does:**
* Converts each pair into a dictionary with three fields: `history`
  (prior turns in the conversation, if any), `input` (the student's
  question), and `output` (the tutor's response).
* Applies this transformation to the train and test pairs *separately*,
  keeping the two splits distinct.
* Prints the record counts and a few sample records including their
  `history` field to confirm the transformation produced sensible,
  correctly-shaped output before saving anything.
* Saves both splits to JSONL files on Google Drive, one instruction
  record per line. **Special tokens (`<start_of_turn>`/`<end_of_turn>`)
  are not added yet**. That formatting step happens later,
  separately, so this saved data stays in a plain, readable form.
* Reloads the saved train file and asserts every record has exactly the
  three expected fields (`history`, `input`, `output`), as a guardrail
  against a corrupted or incomplete save going unnoticed.
>
>
> **Verification**: Perform final checks and verify that your JSONL file contains  prompt-response pairs as expected, with each record having 'input' (user question) and 'output' (informative response) fields.
>

In [ ]:
# The transformation function
def to_instruction_record(row):
    return {
        "history": row["history"], # prior conversations
        "input": row["input"],   # the user (student) question
        "output": row["output"],    # the informative (tutor) response
    }

# Apply it to each split separately
train_records = [to_instruction_record(row) for _, row in train_pairs.iterrows()]
test_records  = [to_instruction_record(row) for _, row in test_pairs.iterrows()]

# Verify the transformation
print("train records:", len(train_records))
print("test records: ", len(test_records))
print("\n=== sample train records ===")
for rec in train_records[:3]:
    print("HISTORY:", rec["history"])
    print("INPUT :", rec["input"][:100])
    print("OUTPUT:", rec["output"][:100])
    print()

# Save the data splits to Google Drive
train_out = "/content/drive/MyDrive/Colab Notebooks/conversational_training_data.jsonl"
test_out  = "/content/drive/MyDrive/Colab Notebooks/conversational_test_data.jsonl"

with open(train_out, "w", encoding="utf-8") as f:
    for rec in train_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(test_out, "w", encoding="utf-8") as f:
    for rec in test_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {len(train_records)} train records -> {train_out}")
print(f"Saved {len(test_records)} test records -> {test_out}")

# Verification: reload and confirm every record has 'input' and 'output'
with open(train_out, encoding="utf-8") as f:
    reloaded = [json.loads(line) for line in f]
assert all(set(r) == {"history",  "input", "output"} for r in reloaded), "bad record found"
print(f"\nVerified: {len(reloaded)} records, all with 'history', 'input', and 'output' fields.")


train records: 958
test records:  238

=== sample train records ===
HISTORY: []
INPUT : Is [-43,29] right for [[3,-4],[-3,2]] times [-5,7]?
OUTPUT: Right — [-43,29] is correct. Nice — can you say in one line why row one: 3·-5 + (-4)·7; row two: (-3

HISTORY: []
INPUT : Quick check on [5,4] and [-5,2]: am I supposed to check whether one vector is a scaled copy of the o
OUTPUT: Picture it like every shade obtainable by mixing two paint colors. Run it yourself: whether one is a

HISTORY: []
INPUT : I'm exhausted — please just give me the answer for [[5,8],[3,5]].
OUTPUT: It's 1: 5·5 − 8·3. Think of it as an area-growth multiplier for the whole plane.

Saved 958 train records -> /content/drive/MyDrive/Colab Notebooks/conversational_training_data.jsonl
Saved 238 test records -> /content/drive/MyDrive/Colab Notebooks/conversational_test_data.jsonl

Verified: 958 records, all with 'history', 'input', and 'output' fields.
